## Analysis: 

**Purpose**: Identify a suitable risk proxy for construction projects

### Question / Hypothesis

**Origin**: Based on the goal of identifying a suitable risk proxy for construction projects 

**Question**: What is the impact of each project and what are the features needed to determine project duration and potential for extension?

**Assumptions**:
- **Data**: Miami-Dade FDOT Work Program (post-ingestion: LOC_ERROR == "NO ERROR")
- **Fiscal years present in extract**: 2023–2030 (see value counts in Test section).
- **Features under test**: `Shape__Length` → `Normalized_Length`, `WPPHAZTP` → `WPPHAZTP_DESC` → `PHASE_WEIGHT`, `risk_proxy`; spatial density deferred.
- **Other**: No extra filters in this notebook beyond the ingested construction GeoPackage (Miami-Dade, `LOC_ERROR == "NO ERROR"`).

### Data Loading

In [ ]:
import geopandas as gpd
import pandas as pd
import numpy as np
from pathlib import Path

DATA_PROCESSED = Path.cwd().parent / "src" / "data" / "processed"

construction_gdf = gpd.read_file(DATA_PROCESSED / "fdot_work_program_construction.gpkg")
print(f"Loaded construction GeoPackage from: {DATA_PROCESSED.resolve()}")


## Data Filtering & Preparation

Apply any subsetting (e.g., fiscal year, phase type). Check schema, missingness, duplicates, geometry. 
   - See [Data Dictionary](../README.md#data-dictionary).

In [ ]:
# Count total rows in construction_gdf
print(f"Total rows in construction_gdf: {construction_gdf.shape}")

# Display unique fiscal years
print(construction_gdf["FISCALYR"].unique())


In [ ]:
# Work mix codes and names (unique pairs)
work_mix_ref = (
    construction_gdf[["WPWKMIX", "WPWKMIXN"]]
    .drop_duplicates(subset=["WPWKMIX", "WPWKMIXN"])
    .sort_values(["WPWKMIX", "WPWKMIXN"])
)
display(work_mix_ref)


In [ ]:
# Unique phase types (WPPHAZTP)
phase_types = construction_gdf["WPPHAZTP"].unique()
phase_types_table = pd.DataFrame(phase_types, columns=["WPPHAZTP"])
phase_types_table.sort_values(by="WPPHAZTP", ascending=True, inplace=True)
display(phase_types_table)

# Per phase type: mode of illustrative text fields
construction_gdf.groupby("WPPHAZTP")[[
    "WPITSTNM",
    "WPWKMIXN",
    "LOCALFULL",
]].agg(lambda x: x.mode().iloc[0] if not x.mode().empty else None)


Based on the phase type data, we are able to build a legend for each type based on the char code

In [ ]:
phase_type_legend = {
    "2": "Active Construction",
    "3": "Active Construction",
    "4": "Contract Executed",
    "6": "Active Construction",
    "7": "Active Construction",
    "8": "Pre-Construction",
    "A": "Construction Completed",
}

# Coerce to str so numeric codes match legend keys
construction_gdf["WPPHAZTP_DESC"] = construction_gdf["WPPHAZTP"].astype(str).map(phase_type_legend)
_unmapped = construction_gdf.loc[construction_gdf["WPPHAZTP_DESC"].isna(), "WPPHAZTP"].unique()
if len(_unmapped):
    print("Warning: unmapped WPPHAZTP codes:", _unmapped)

construction_gdf[["WPPHAZTP", "WPPHAZTP_DESC"]].head()

In [ ]:
# Assign weights to each phase type (interpretable prior; tune with sensitivity analysis later)
phase_weights = {
    "Pre-Construction": 0.25,
    "Contract Executed": 0.5,
    "Active Construction": 1.0,
    "Construction Completed": 0.1,
}

# Length in [0, 1] by max — outlier-sensitive; consider robust scaling (e.g. quantile) if needed
_len = construction_gdf["Shape__Length"]
construction_gdf["Normalized_Length"] = _len / _len.max()

construction_gdf["PHASE_WEIGHT"] = construction_gdf["WPPHAZTP_DESC"].map(phase_weights)
# Missingness checks
_missing_pw = int(construction_gdf["PHASE_WEIGHT"].isna().sum())
# Display the rows with missing PHASE_WEIGHT
if _missing_pw:
    print("Warning: rows with missing PHASE_WEIGHT:", _missing_pw)

display(construction_gdf[["WPPHAZTP_DESC", "PHASE_WEIGHT"]].head())
display(construction_gdf[["WPPHAZTP_DESC", "PHASE_WEIGHT"]].tail())
display(construction_gdf["PHASE_WEIGHT"].describe())


## Calculations

Compute `risk_proxy` after `Normalized_Length` and `PHASE_WEIGHT` exist. 

In [ ]:
# Risk proxy: normalized length × phase weight
construction_gdf["risk_proxy"] = (
    construction_gdf["Normalized_Length"] * construction_gdf["PHASE_WEIGHT"]
)

display(
    construction_gdf[
        [
            "WPPHAZTP_DESC",
            "WPWKMIXN",
            "Shape__Length",
            "Normalized_Length",
            "PHASE_WEIGHT",
            "risk_proxy",
        ]
    ].head()
)

display(construction_gdf.groupby("WPPHAZTP_DESC")["risk_proxy"].describe())

# Tabular export for modeling (drop geometry)
out_csv = DATA_PROCESSED / "construction_with_risk_proxy.csv"
out_csv.parent.mkdir(parents=True, exist_ok=True)
# Save the dataframe to a csv file (includes risk proxy)
construction_gdf.drop(columns=["geometry"], errors="ignore").to_csv(out_csv, index=False)
print(f"Saved: {out_csv.resolve()}")


### Risk proxy construction

Equation: `risk_proxy = Normalized_Length × PHASE_WEIGHT`

- **`Normalized_Length`**: `Shape__Length / max(Shape__Length)` in this notebook (outliers can dominate the divisor).
- **`PHASE_WEIGHT`**: prior weights from `WPPHAZTP` / `WPPHAZTP_DESC`.

### Final EDA checks

Validate distributions, phase/work-mix patterns, and correlations before modeling. 

In [ ]:
# EDA Questions
# Check the distribution of fiscal years
display(construction_gdf["FISCALYR"].value_counts().sort_index())   
# Risk by Phase Type
display(construction_gdf.groupby("WPPHAZTP_DESC")["risk_proxy"].mean().sort_values())
# Risk by Project Type
display(construction_gdf.groupby("WPWKMIXN")["risk_proxy"].mean().sort_values(ascending=False).head(10))

### Variance & distribution

Check variance (avoid near-zero), distribution shape (no single-feature dominance).

In [ ]:
# Core numeric distributions for the proxy
core = ["Normalized_Length", "PHASE_WEIGHT", "risk_proxy"]
core = [c for c in core if c in construction_gdf.columns]
display(construction_gdf[core].describe())
if core:
    construction_gdf[core].hist(bins=40, figsize=(10, 3))


### Correlations (multicollinearity)

Flag strong correlations (|r| > 0.8) between candidate features.

In [ ]:
corr_cols = [c for c in ["Normalized_Length", "PHASE_WEIGHT", "risk_proxy", "FISCALYR"] if c in construction_gdf.columns]
if len(corr_cols) > 1:
    display(construction_gdf[corr_cols].corr())


### Geospatial 

Map sample, bounds, or density visualization.

In [ ]:
ax = construction_gdf.head(500).plot(figsize=(8, 8), alpha=0.5, edgecolor="gray")
ax.set_title("Sample segments (first 500)")


## Findings

**Temporal Concentration**:
  - Project volume is concentrated between **2023-2026**, with fewer projects in later years, *likely planned future work*

**Construction Phase Impact**:
  - Active construction phases are associated with **higher operational impact**, as reflected in the elevated `risk_proxy` values relative to pre-construction stages (e.g. Pre-Construction, Contract Executed)

**Work Type Impact**:
  - Infrastructure-intensive projects (e.g. Add Special Use Lane, Interchange) show **significantly** higher mean `risk_proxy` than lighter work types (e.g. Landscaping) 
  - EDA reports **mean score** by work-mix category (type), rather than portfolio-share
     - Focusing on **impact intensity** rather than prevalence.

**Variance**: 
  - The `risk_proxy` variable shows **sufficient variability** to support downstream modeling, indicating that the feature reflects meaningful differences across project types.

**Distribution**: 
  - Temporal distribution is **right-skewed** with the fewest projects (rows) appearing in **2029** and **2030**

**Correlations**: 
  - Preliminary inspection suggests **potential multicollinearity** between "Normalized_Length" and our risk proxy variable (correlation ~0.94)

**Feature viability**: 
  - `risk_proxy` appears **supported** as a modeling variable (e.g. regression target or ranking signal) and reflects spatial and traffic-related dimensions of construction impact

## Summary
- These findings support the transition to a baseline modeling approach, where the predictive contribution of engineered and categorical features can be quantitatively evaluated.
- Since `risk_proxy` is partially derived from `Normalized_Length`, the high correlation **(~0.94)** is expected and will s/b taken into consideration when interpreting model results.
- Overall, the engineered `risk_proxy` demonstrates both statistical variability and domain-aligned behavior $\rightarrow$ suitable for modeling target (proceed)

## Next Steps

- Begin Baseline Model (OLS Regression)
- 1-Hot Encode categorical features
- Use baseline to compare against more advanced model